# Importing Libraries

In [ ]:
# Installing necessary packages
!pip install pandas numpy matplotlib seaborn scipy scikit-learn plotly

# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats

print("Libraries imported successfully!")

# Uploading Dataset

In [ ]:
# If running in Google Colab, this lets you upload season-2526.csv from your computer.
# If running locally in Jupyter, skip this cell and just make sure season-2526.csv
# is in the same folder as this notebook, then run the cell below instead.
from google.colab import files
uploaded = files.upload()

import pandas as pd
import io

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Data imported successfully!")
print(f"Shape: {df.shape}")
df.head()

# Statistical Analysis of Performance Metrics

In [ ]:
import pandas as pd
from scipy import stats

try:
    # --- Step 1: Load Data ---
    file_path = 'season-2526.csv'
    df = pd.read_csv(file_path)

    # --- Step 2: Data Transformation (Calculate League Table) ---

    # Get all unique team names
    teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()

    # Initialize dictionaries to store points and goal difference
    points = {team: 0 for team in teams}
    gd = {team: 0 for team in teams}

    # Iterate through each match to calculate points and GD
    for index, row in df.iterrows():
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']
        home_goals = row['FTHG']
        away_goals = row['FTAG']

        # Calculate Goal Difference
        gd[home_team] += home_goals - away_goals
        gd[away_team] += away_goals - home_goals

        # Assign Points
        if row['FTR'] == 'H':
            points[home_team] += 3
        elif row['FTR'] == 'A':
            points[away_team] += 3
        elif row['FTR'] == 'D':
            points[home_team] += 1
            points[away_team] += 1

    # Create League Table DataFrame
    league_table = pd.DataFrame({'Points': points, 'GD': gd})

    # Sort by Points (desc), then GD (desc)
    league_table = league_table.sort_values(by=['Points', 'GD'], ascending=[False, False])

    # Add League Position column
    league_table['League_Position'] = range(1, len(league_table) + 1)

    # --- Step 3: Data Transformation (Aggregate Team Stats) ---

    # Calculate Total Clean Sheets per team
    home_cs = df[df['FTAG'] == 0].groupby('HomeTeam').size().rename('Home_CS')
    away_cs = df[df['FTHG'] == 0].groupby('AwayTeam').size().rename('Away_CS')
    total_cs = pd.concat([home_cs, away_cs], axis=1).fillna(0).sum(axis=1).rename('Total_Clean_Sheets')

    # Calculate Total Cards per team (Yellow + Red)
    home_cards = (df.groupby('HomeTeam')['HY'].sum() + df.groupby('HomeTeam')['HR'].sum()).rename('Home_Cards')
    away_cards = (df.groupby('AwayTeam')['AY'].sum() + df.groupby('AwayTeam')['AR'].sum()).rename('Away_Cards')
    total_cards = pd.concat([home_cards, away_cards], axis=1).fillna(0).sum(axis=1).rename('Total_Cards')

    # --- Step 4: Merge Aggregated Data ---

    # Merge all stats into one DataFrame, indexed by team name
    team_stats = league_table.merge(total_cs, left_index=True, right_index=True)
    team_stats = team_stats.merge(total_cards, left_index=True, right_index=True)

    # --- Step 5: Run Statistical Tests ---

    # Test 1: Clean Sheets vs. League Position
    cs_corr = stats.pearsonr(team_stats['Total_Clean_Sheets'], team_stats['League_Position'])

    # Test 2: Total Cards vs. League Position
    cards_corr = stats.pearsonr(team_stats['Total_Cards'], team_stats['League_Position'])

    # Test 3: Home Advantage (Paired t-test on the original 380-match dataframe)
    home_adv = stats.ttest_rel(df['FTHG'], df['FTAG'])

    # --- Step 6: Print Results ---

    print("--- Aggregated Team Stats (Top 5) ---")
    print(team_stats.head())
    print("\n")

    print("--- Statistical Test Results (for your paper) ---")
    print(f"1. Clean Sheets vs. League Position: r = {cs_corr[0]:.4f}, p = {cs_corr[1]:.4f}")
    print(f"2. Total Cards vs. League Position:     r = {cards_corr[0]:.4f}, p = {cards_corr[1]:.4f}")
    print(f"3. Home Advantage (Home vs. Away Goals): t = {home_adv.statistic:.4f}, p = {home_adv.pvalue:.4f}")

except FileNotFoundError:
    print(f"Error: The file 'season-2526.csv' was not found.")
except KeyError as e:
    print(f"Error: A required column {e} was not found in the CSV.")
    print("Please ensure your CSV contains columns like 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HY', 'AY', 'HR', 'AR'.")